# makemore part 5 — Building a WaveNet（观后笔记）

**本章只看了视频，没有跟敲，仓库里没有可运行的代码。**
留这份笔记是为了以后回头找层次结构和那个 BatchNorm 坑时有个着落点。

这一章没有新数学。做的是两件事：一次重构，一次结构升级。


## 1 — 把零散张量收成 layer

03/04 章的网络是一堆裸露的张量：`C`、`W1`、`b1`、`bngain`… 前向手写在训练循环里，
加一层就要改三处。这一章照着 PyTorch 的接口把它们收成类：

```
Linear          weight / bias，__call__ 做 x @ W + b
BatchNorm1d     gamma / beta + running_mean / running_var，training 开关
Tanh            无参数
Embedding       就是 C[x] 这一步
Flatten         就是 .view(N, -1) 这一步
Sequential      把上面串起来
```

每个类提供 `__call__` 和 `parameters()`。收完之后整个前向塌成一行 `model(x)`，
训练循环里再也看不到具体有几层。

这不是美化——**它是后面能随手换结构的前提**。第 3 节要把扁平换成层次，
如果前向还是手写的，那就是重写；有了 `Sequential` 只是换一个列表。


## 2 — FlattenConsecutive：从扁平到层次

**扁平版（03/04 章的做法）**：8 个字符一次性拼成 `(B, 8*n_embd)`，
第一个 `Linear` 一层就把全部上下文吃完。

```
(B, 8, C)  --Flatten-->  (B, 8C)  --Linear-->  (B, H)
```

问题：8 个字符的信息在第一层就被压成一个向量，中间的组合结构全丢了。

**层次版（WaveNet 的骨架）**：每次只合并**相邻两个**，逐层收拢。

```
(B, 8, C)  -->  (B, 4, 2C)  -->  (B, 2, 4C)  -->  (B, 1, 8C)
       两两合并        两两合并        两两合并
```

`FlattenConsecutive(n)` 做的就是把 `(B, T, C)` 变成 `(B, T//n, n*C)`。
每一层之间夹 `Linear + BatchNorm1d + Tanh`，于是字符对先融合成"双字符特征"，
再融合成"四字符特征"，最后才汇总。**这就是 WaveNet 论文里那棵树**，
只不过论文用的是空洞卷积，这里用 reshape 实现同一件事。

一句话：扁平版是一次性全连接，层次版是逐层二分融合。


## 3 — BatchNorm1d 的三维 bug（本章唯一的真坑）

层次结构一上来，进 BN 的张量就从二维 `(B, C)` 变成三维 `(B, T, C)` 了。

原来的实现在 `dim=0` 上求统计量。三维输入下这么写：

- **不会报错**
- 训练照常收敛
- 但 `running_mean` 的形状是 `(1, T, C)` 而不是 `(1, C)`

意思是每个时间位置各自维护了一套统计量——而 BatchNorm 的定义是
**每个通道一套**。正确写法是在 `dim=(0, 1)` 上求：

```python
dim = 0 if x.ndim == 2 else (0, 1)
xmean = x.mean(dim, keepdim=True)
xvar  = x.var(dim, keepdim=True)
```

Karpathy 在视频里专门停下来讲了这个，因为它是**静默的**：
loss 曲线看不出异常，只有去看 buffer 形状才发现。

PyTorch 自己的 `nn.BatchNorm1d` 对三维输入的约定是 `(N, C, L)`——通道在**中间**，
和这里的 `(B, T, C)` 不一样。混用会错得更隐蔽。


## 4 — 和 07 的关系

这一章把上下文从 3 推到 8，靠的是"堆更深、每层融合更少"。
但它仍然是**固定窗口**：超出 `block_size` 的历史彻底看不见，
而且每个位置怎么融合是写死在结构里的。

第 07 章的 self-attention 换掉的正是这一点——
让每个位置**自己决定**去看历史里的哪些位置，权重由数据算出来而不是结构定死。
